In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import linregress
from plotly.subplots import make_subplots

### The relationship between cloud and weather factors

In [2]:
df = pd.read_csv("img_attribute.csv")
df['TotalCloud_weighted'] = df[['Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted',
                                'Stratocumulus_weighted', 'Cumulus_weighted', 'Cirrocumulus_weighted',
                                'Nimbus_weighted']].sum(axis=1)
features = ['Cirrus_weighted', 'Cirrostratus_weighted', 'Stratus_weighted', 'Stratocumulus_weighted',
            'Cumulus_weighted', 'Cirrocumulus_weighted', 'Nimbus_weighted', 'TotalCloud_weighted', 'Clear_weighted']
df[features] = df[features].round(4)
colors = [
    '#FF4B4B', '#FF9600', '#FFD24B',
    '#AFE182', '#82AFE1', '#AF96FF',
    '#969696', '#555555', '#000000',
]

In [3]:
y_var = 'Radiation Value'
# y_var = 'Temperature(C)'
# y_var = 'Absolute_Humidity(g/m^3)'

fig = make_subplots(
    rows=3, cols=3,
    shared_xaxes=False, shared_yaxes=False,
    horizontal_spacing=0.03, vertical_spacing=0.09,
    subplot_titles=[f.replace('_weighted', '') for f in features]
)

for idx, x_var in enumerate(features):
    row = (idx // 3) + 1
    col = (idx % 3) + 1
    x = df[x_var]
    y = df[y_var]
    mask = x > 0.01
    x = x[mask]
    y = y[mask]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode='markers',
            marker=dict(size=5, opacity=0.9, color=colors[idx]),
            name=f'{x_var} vs {y_var}',
            showlegend=False
        ),
        row=row, col=col
    )
    temp_df = pd.DataFrame({'x': x, 'y': y}).dropna()
    x_clean = temp_df['x']
    y_clean = temp_df['y']
    if len(x_clean) > 1:
        slope, intercept, r_value, p_value, std_err = linregress(x_clean, y_clean)

        x_range = np.array([x_clean.min(), x_clean.max()])
        y_range = slope * x_range + intercept
        
        fig.add_trace(
            go.Scatter(
                x=x_range,
                y=y_range,
                mode='lines',
                line=dict(color='dimgrey', width=2),
                name='Regression Line'
            ),
            row=row,
            col=col
        )
        fig.add_annotation(
            x=0.98, y=0.05,
            xref="x domain",
            yref="y domain",
            text=f"Slope: {slope:.2f}, p-value: {p_value:.2f}",
            showarrow=False,
            row=row, col=col,
            align='right',
            font=dict(size=18),
            bgcolor="rgba(255, 255, 255, 0.5)"
        )

    fig.update_xaxes(range=[-0.05, 1.05], row=row, col=col)
    fig.update_yaxes(title_text=None, row=row, col=col)

if y_var == 'Radiation Value':
    fig.update_yaxes(range=[-59, 10])
elif y_var == 'Temperature(C)':
    fig.update_yaxes(range=[0, 30])    
elif y_var == 'Absolute_Humidity(g/m^3)':
    fig.update_yaxes(range=[-2, 25])

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f'Weighted Cloud (X) vs. {y_var} (Y)',
        x=0.5,
        xanchor='center',
        font=dict(size=22)
    ),
    showlegend=False,
    margin=dict(r=20, l=40, t=80, b=40),
    template='simple_white'
)
fig.show()

### The relationship between cloud and V channel

In [4]:
df = pd.read_csv("hsv_mean.csv")
df.rename(columns={'v_mean': 'V_mean'}, inplace=True)

In [5]:
feature = 'V_mean'
target = 'Radiation Value'
x, y = df[feature], df[target]

# calculate correlation coefficient and R²
slope, intercept, r_value, p_value, _ = linregress(x, y)
correlation_coefficient = r_value
r2 = r_value**2

# regression line
x_sorted = np.sort(x)
y_fit = slope * x_sorted + intercept

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode='markers',
        marker=dict(size=15, opacity=0.6, color='#39BCBC'),
        name='Data',
        showlegend=False
    )
)

fig.update_layout(
    height=1200,
    width=1400,
    xaxis = dict(
        title = dict(text='Average V value of the image', standoff=30),
        tickfont=dict(size=56, family='Arial', weight='bold'),
        titlefont=dict(size=64, family='Arial', weight='bold'),
        linewidth=6, linecolor='black', tickwidth=6, ticklen=15
    ),
    yaxis = dict(
        title = dict(text='NLW Radiation (W·m⁻²)', standoff=30),
        tickfont=dict(size=56, family='Arial', weight='bold'),
        titlefont=dict(size=64, family='Arial', weight='bold'),
        linewidth=6, linecolor='black', tickwidth=6, ticklen=15
    ),
    margin=dict(t=10, b=180, l=220, r=10),
    template='simple_white'
)

fig.show()